# Gulfstream walkthrough — yield curves (`zero_rates`)

This notebook runs the **full Graph 1 core path** on real curve data:

1. Load `zero_rates` (+ optional FX) from DuckDB
2. Engineer curve / FX features
3. Reduce dimension (PCA) → random Fourier features → ruptures candidates
4. Validate breakpoints with an MMD test
5. Inspect regimes with plots

**Database:** `D:/data/duckdb/ycs_data.duckdb` · **table:** `zero_rates`

Prefer running cells top-to-bottom. After you finish, tell the agent so outputs can be verified.


## 0. Project setup

Add the package to the path (works whether you launched Jupyter from the repo root or `notebooks/`).


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import duckdb
import pandas as pd
import polars as pl
from plotnine import aes, geom_line, ggplot, labs, theme_bw, facet_wrap, theme

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "pyproject.toml").exists():
    ROOT = NOTEBOOK_DIR
elif (NOTEBOOK_DIR.parent / "pyproject.toml").exists():
    ROOT = NOTEBOOK_DIR.parent
else:
    ROOT = Path(r"D:/Code/gulfstream")

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

YCS_DB = Path(r"D:/data/duckdb/ycs_data.duckdb")
OUT_DIR = ROOT / "outputs" / "notebooks" / "ycs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT =", ROOT)
print("YCS_DB exists =", YCS_DB.exists())


## 1. Peek at `zero_rates`

Long format: one row per `(date, source)` with tenor columns `Y002p0`, `Y005p0`, …


In [ ]:
con = duckdb.connect(str(YCS_DB), read_only=True)
print("tables:", con.execute("SHOW TABLES").fetchall())
sample = con.execute(
    """
    SELECT date, source, Y002p0, Y005p0, Y010p0, Y030p0
    FROM zero_rates
    WHERE source IN ('USA', 'DEU', 'ITA')
      AND date >= '2015-01-01'
    ORDER BY date, source
    LIMIT 6
    """
).pl()
print(sample)
coverage = con.execute(
    """
    SELECT source, COUNT(*) AS n, MIN(date) AS dmin, MAX(date) AS dmax
    FROM zero_rates
    GROUP BY 1
    ORDER BY 1
    """
).pl()
print(coverage)
con.close()


## 2. Load through gulfstream

`config/sources/notebook_ycs.yaml` pivots USA/DEU/ITA tenors, joins EURUSD/GBPUSD, then runs
`generate_yield_features` (spreads, butterflies, rolling vol / corr).


In [ ]:
from gulfstream.common import frames, utils
from gulfstream.pipelines.hamilton.driver import load_features, run_segmentation_pair
from gulfstream.detection import time_index as bkpt_time
from gulfstream.metrics import regime_plots

features_df, source_type = load_features(
    ROOT / "config" / "sources" / "notebook_ycs.yaml",
    project_root=ROOT,
)
print("source_type:", source_type)
print("shape:", features_df.shape, "n_features:", frames.n_features(features_df))
print("date range:", features_df["date"].min(), "→", features_df["date"].max())
print("feature sample:", frames.feature_columns(features_df)[:10])
features_df.head(3)


## 3. Explore a few series

Plot selected outright / derived features before detection.


In [ ]:
plot_cols = [
    c
    for c in [
        "USA_Y010p0",
        "DEU_Y010p0",
        "ITA_Y010p0",
        "USA_Y002p0_minus_USA_Y010p0",
        "EURUSD",
    ]
    if c in features_df.columns
]
long = (
    features_df.select(["date", *plot_cols])
    .unpivot(index="date", on=plot_cols, variable_name="series", value_name="value")
    .to_pandas()
)
long["date"] = pd.to_datetime(long["date"])

(
    ggplot(long, aes("date", "value", color="series"))
    + geom_line(size=0.4)
    + facet_wrap("~series", scales="free_y", ncol=1)
    + theme_bw()
    + theme(figure_size=(10, 2.2 * len(plot_cols)), legend_position="none")
    + labs(title="Selected yield / FX features", x="", y="")
)


## 4. Algorithm config

Reuse `config/graph1/default_core.yaml`: PCA → RFF → ruptures → MMD, with plots in **display** mode.


In [ ]:
params = utils.read_config_yaml(
    str(ROOT / "config" / "graph1" / "default_core.yaml"),
    img_dir=str(OUT_DIR),
    log_dir=str(ROOT / "outputs" / "logs"),
)
params["test_num"] = 0
params["metrics"]["mode"] = "display_and_write"
params["metrics"]["plot"] = True
params["metrics"]["dir"] = str(OUT_DIR)
params["metrics"]["image_dir"] = str(OUT_DIR)
params["metrics"]["features_to_plot"] = plot_cols[:3]
params["robustness"]["enabled"] = False
params["stability"]["enabled"] = False

print("dimred:", params["algo"]["dimred"])
print("test:", params["test"]["choice"])
print("depth / min_regime:", params["algo"]["depth"], params["algo"]["min_regime_length"])


## 5. Run Hamilton single-pass segmentation

This is the same linear core Graph 1 / Graph 2 use: dimred → feature map → ruptures → postprocess.


In [ ]:
unprocessed, processed = run_segmentation_pair(features_df, params)

print("raw breakpoints (indices):", unprocessed.bkpts)
print("kept breakpoints:", processed.bkpts)
print("invalid / rejected:", processed.invalid_bkpts)
print("low confidence:", processed.low_confidence_bkpts)

dates = frames.dates_series(features_df).to_list()
for b in processed.bkpts:
    print(f"  bkpt {b} → {dates[b]}")

hierarchy = processed.hierarchy or {b: 1 for b in processed.bkpts}
regimes_df = bkpt_time._get_regime_intervals(hierarchy, dates)
regimes_df


## 6. Regime visualization

Shaded intervals + vertical lines at accepted breakpoints.


In [ ]:
regime_plots.produce_all_regime_visualization_tools(features_df, params, processed)
print("Plots also written under", OUT_DIR)


## 7. Compact custom regime chart

Same idea as the gallery plot, built inline with plotnine for notebook display.


In [ ]:
from gulfstream.metrics.regime_plots import _visualize_market_regimes

_visualize_market_regimes(
    features_df,
    regimes_df,
    title="YCS regimes (notebook)",
    variables=plot_cols[:2],
    valid_bkpts=processed.bkpts,
    invalid_bkpts=processed.invalid_bkpts,
    low_confidence_bkpts=list(processed.low_confidence_bkpts or []),
    mode="display",
)


## 8. Equivalent CLI

Once you are happy with the YAML knobs:

```bash
uv run python -m gulfstream.cli --mode graph1 \
  --config config/graph1/default_core.yaml \
  --source-config config/sources/notebook_ycs.yaml \
  --img-dir outputs/metrics
```

Next notebook (`02_equity_eod_workflow.ipynb`) repeats the same detection path on equity closes.
